# nib — Colab evaluation

**This notebook contains no logic.** It clones, installs, mounts Drive, copies
one file to local disk, and calls scripts. Every decision lives in
`configs/base.yaml` and every line of code lives in the repository.

## What this run is for

On 2026-09-11 the project's style metric was found to be measuring the wrong
thing. A **real** line, by unquestionably the right writer, blurred by 0.8
pixels — damage that does not change whose handwriting it is — scored 12.2%
where the untouched line scored 96.8%. Every generative decoder produces
exactly that softness, so writer retrieval has been reporting sharpness as much
as style, and every style comparison this project has made is confounded by it.

`HWD` replaces it. It barely moves under the same damage, it separates
handwriting from a typeface by a wide margin, and it is what Emuru's and Eruku's
own papers report.

**Its scale is measured inside the run, not quoted.** HWD compares each writer's
*average* look, and an average over few lines is noisy, so what real handwriting
scores depends on how many lines each writer contributes. Measured on our data:

```
lines per writer     1      2      3      6      10
real vs real       1.76   1.26   1.01   0.70   0.54
```

The 0.641 quoted earlier sits between six and ten lines per writer. A 300-sample
run gives about three, where real handwriting itself scores about 1.06. So every
run now reports three figures against one shared set of real lines:

- **generated** — what the model wrote
- **real** — the target lines themselves: same writers, same texts, same counts
- **typeface** — the same texts in a font: no hand at all

and, for each, whether it is closer to its own writer than to everyone else —
the **identity** block, which tells "not this writer" apart from "not real
handwriting".

## Run cells 1 to 6 in order

Cell 6 saves its own run to Drive. Cell 7b (two style lines), 7c (quality
control) and cell 8 (Eruku) are optional and save themselves too; cell 7 copies everything again if a run
did not. Anything below cell 9 is kept for reference and should not be run
without a reason.

## Before you start

Under `MyDrive/nib/` you need `cvl_lines_64.lmdb` (**127 MB, 9,142 records** —
the copy from `data/processed/upload/`, never the 8 GB one) and
`checkpoints/writer_embedder.pt`.

## 1. Clone and install

`torch` is deliberately absent — Colab's build is matched to its CUDA driver.

The `hwd` extra is new and it is a large install: the package imports every
score it owns at import time, so it pulls in gudhi, matplotlib, tiktoken and
scikit-learn. Two or three minutes.

**Expect a pip conflict warning about `gradio`.** Installing `transformers<5`
pulls `huggingface-hub` down to the range it needs and Colab's preinstalled
gradio wants something newer. Nothing here imports gradio. Cell 2 is the check
that decides.

In [ ]:
REPO_URL = "https://github.com/omritzabari/nib.git"

%cd /content
![ -d nib ] || git clone $REPO_URL nib
%cd /content/nib
!git pull --ff-only
!pip install -q -e ".[dev,track,models,hwd]"

## 2. What are we running on

**Stop here if any of these is wrong.** A rebuilt Colab VM arrives without an
accelerator unless one is asked for, and generation on CPU is 220 seconds a
line — 18 hours for 300.

- `Tesla T4` must appear
- `torch` must say `+cu128`, not `+cpu`
- `hwd available: True` and `hwd imports cleanly` — otherwise HWD is skipped,
  and the second of those two lines prints the reason instead

In [ ]:
!python --version
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

import torch
import transformers

print("torch       ", torch.__version__, "| cuda", torch.version.cuda)
print("transformers", transformers.__version__, " <- must be 4.x")
assert transformers.__version__.startswith("4."), "5.x cannot load Emuru; see pyproject"

# In a fresh interpreter, not in this kernel: the kernel started before the
# install and has /content on its path, where the clone reads as an empty package
# named `nib`. The scripts always run in a fresh interpreter, so this checks them.
!cd /content/nib && python -c "from nib.engine.metrics import hwd; print('hwd available:', hwd.available(), ' <- must be True')"
!cd /content/nib && python -c "import hwd.scores" && echo "hwd imports cleanly"

## 3. Mount Drive and copy what the run needs

One sequential copy of one file. Reading the pack record-by-record over Drive
would leave the GPU waiting on network round-trips.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

!mkdir -p /content/nib/data/processed /content/nib/checkpoints
!time cp /content/drive/MyDrive/nib/cvl_lines_64.lmdb /content/nib/data/processed/
!cp /content/drive/MyDrive/nib/checkpoints/writer_embedder.pt /content/nib/checkpoints/
!ls -lh /content/nib/data/processed/ /content/nib/checkpoints/

## 4. Is everything here

Two things will read as missing and both are correct: the **word pack**,
because one pack is required and this session needs lines; and the **raw CVL
images**, because 5 GB of sources are not copied to a VM that only reads a
127 MB pack. The only consequence is that CER cannot be re-measured here, and
it has already been measured where the sources are.

In [ ]:
%cd /content/nib
!python scripts/check_data.py

## 5. Check the harness

The `fake` generator draws the target text in a typeface. Every number it
produces is meaningless and every shape is right, which is what a pipeline
check needs. It has caught an unexercised code path twice.

The first run on a fresh VM downloads HWD's VGG weights, 675 MB.

**What must appear**, or something below is not wired:

- square brackets after every figure — the 95% spreads
- an `HWD` block with `generated`, `real` and `typeface` lines — not
  `not measured`, not `FAILED`
- `generated` equal to `typeface`, and `100%` of the way — for this generator
  they are the very same images, so anything else is a bug
- in the `identity` block, `generated` and `typeface` identical again with a gap
  near zero, and `real` clearly above both
- a line beginning `generated` with a folder, and one beginning `analysis`

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py --generator fake --samples 120 --device cuda

## 6. Emuru with a working style metric — the main run

Generation took 53.9 minutes for 300 lines on the last run, then the metrics.

Two blocks under `HWD` matter:

- **distance** — `generated` against `real` and `typeface` from the same run.
  Emuru read 2.00 against 0.86 and 2.99: 53% of the way to no hand at all.
- **identity** — whether each set is closer to *its own* writer than to the
  others. Distance alone cannot tell "not this writer" from "not real
  handwriting"; this can. The last line gives the share of the real lines'
  identity the generated lines carry: 0% is nobody's hand in particular, 100% is
  as distinct as the writer's own lines. `typeface` should sit near a gap of zero.

`truncated` sat at 4.4% last time.

**The cell saves to Drive by itself when the run ends** and prints how many
generated images arrived — about 295.

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py \
    --generator emuru \
    --samples 300 \
    --device cuda

# To Drive the moment the run ends. The VM that produced the first trustworthy
# style figure was reclaimed before anyone copied it.
!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/eval_emuru_lines /content/drive/MyDrive/nib/results/
!echo "saved to Drive: $(ls /content/drive/MyDrive/nib/results/eval_emuru_lines/generated | wc -l) generated images"

## 7. Save everything again — only if a run above did not

Cells 6, 7b and 8 save their own run to Drive when they finish. This copies
every run and the references, and is safe to repeat.

In [ ]:
!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/eval_* /content/drive/MyDrive/nib/results/
!cp -r /content/nib/references /content/drive/MyDrive/nib/results/
!du -sh /content/drive/MyDrive/nib/results/*
!for f in /content/nib/outputs/eval_*/results.json; do echo "== $f"; cat "$f"; done

## 7b. Emuru with two style lines — optional, about as long as cell 6

The model is given one line of a hand and asked to learn it; a real user brings
a page. Two lines generated normally on 2026-09-10 (0.85x of real width) while
four broke Emuru's stopping rule, so two is the most evidence it can take as it
stands.

Compare its `identity` line with cell 6's. If the intervals separate, more
evidence of the hand helps, and fixing the stopping rule to allow more lines is
worth the work. If they overlap, it does not, at least at two.

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py \
    --generator emuru \
    --samples 300 \
    --style-refs 2 \
    --device cuda

!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/eval_emuru_lines_refs2 /content/drive/MyDrive/nib/results/
!echo "saved to Drive: $(ls /content/drive/MyDrive/nib/results/eval_emuru_lines_refs2/generated | wc -l) generated images"

## 7c. Quality control — several draws per line, the broken ones rejected

Emuru's output is bimodal: the typical line reads well, and about one in ten
collapses into a smear or comes out blank. This draws up to four candidates per
line, each from **one** of four style lines by that writer, reads each with
TrOCR-small, and keeps the first readable one. TrOCR-base still measures CER, so
the selector is not also the judge.

The first command is a check on the fake generator and takes a few minutes, most
of it downloading TrOCR-small and TrOCR-base. The Emuru run starts only if the
check succeeds. How long the Emuru run takes depends on how many lines need a
second draw; the `selection` line reports it.

**Compare with cell 6** — identity 56.5% [50.9, 61.9], CER 30.4%, FID 67.70:

- `HWD identity` and `FID` are the clean measures; no recogniser touches them
- `selection` says how many draws were spent
- `CER` should fall, but it is partly flattered by selection

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py --generator fake --samples 60 --style-refs 4 --candidates 2 --device cuda \
  && python scripts/evaluate_generator.py --generator emuru --samples 300 --style-refs 4 --candidates 4 --device cuda

!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/eval_emuru_lines_refs4_cand4 /content/drive/MyDrive/nib/results/
!echo "saved to Drive: $(ls /content/drive/MyDrive/nib/results/eval_emuru_lines_refs4_cand4/generated | wc -l) generated images"

## 8. Eruku, for the comparison — optional, 128 minutes last time

Eruku runs at 35 seconds a line: classifier-free guidance is two forward passes
per token rather than one, which is what buys its text fidelity and what costs
the time.

Worth doing because the Emuru-against-Eruku comparison made on 2026-09-10 was
decided by a metric that measures sharpness, and Eruku's output may simply be
softer. HWD's distance and identity blocks will say. The cell saves itself to
Drive.

In [ ]:
%cd /content/nib
!python scripts/evaluate_generator.py \
    --generator eruku \
    --samples 300 \
    --device cuda

!mkdir -p /content/drive/MyDrive/nib/results
!cp -r /content/nib/outputs/eval_eruku_lines /content/drive/MyDrive/nib/results/
!echo "saved to Drive: $(ls /content/drive/MyDrive/nib/results/eval_eruku_lines/generated | wc -l) generated images"

## 9. Compare what ran — no GPU, seconds

Reads what each run saved and reports whether the intervals separate at all.
Two figures whose intervals overlap are not a difference.

In [ ]:
%cd /content/nib
!python scripts/analyse_run.py outputs/eval_emuru_lines outputs/eval_eruku_lines

---

# Kept for reference — do not run without a reason

**Re-measuring the references** (`check_metrics.py`). They are measured and
committed in `references/`. CER cannot be computed here anyway, and the cell
downloads 1.4 GB of TrOCR in order to skip it.

**The guidance sweep.** Closed negative on 2026-09-10: cfg 1.0 / 1.25 / 2.0 gave
retrieval 10.0% / 5.3% / 6.7% with every interval overlapping, and cfg 2.0 was
clearly worse (15% truncated). Even cfg 1.0's upper bound sat below Emuru. It
would be worth revisiting only once HWD has replaced the metric that judged it.

**More than one style line.** Two are measured in cell 7b. `--style-refs 4`
breaks Emuru (0.25x of real width) because the prefix outgrows what its stopping
heuristic tolerates; the way to more lines is fixing that rule, not this flag.

In [ ]:
# %cd /content/nib
# !python scripts/check_metrics.py --pack data/processed/cvl_lines_64.lmdb --samples 300 --device cuda
# !python scripts/evaluate_generator.py --generator eruku-no-style-text --samples 300 --device cuda

## What to report back

- the `SUMMARY` block from each run, **with its intervals**
- the whole `HWD` block — the distance lines, the percentage, and the
  `identity` block with its last line. This is the one that matters
- `truncated` and `empty outputs`
- the `saved to Drive` line
- anything that failed, with the full error text